## 2.1 理论计算题
输入：$C_{in} \times H_{in} \times W_{in} = 3 \times 32 \times 32$，卷积核数量$K=16$，卷积核尺寸$3\times5\times5$，$Padding=2$，$Stride=2$

### 1. 计算卷积输出特征图尺寸
卷积输出高、宽公式：
$$
H_{out} = \left\lfloor \frac{H_{in} + 2\times Padding - Kernel_{size}}{Stride} \right\rfloor + 1
$$
$$
W_{out} = \left\lfloor \frac{W_{in} + 2\times Padding - Kernel_{size}}{Stride} \right\rfloor + 1
$$
输出通道数 = 卷积核个数。

代入计算：
$$
H_{out} = \lfloor \frac{32 + 2\times2 - 5}{2} \rfloor + 1 = 16
$$
$$
W_{out} = \lfloor \frac{32 + 2\times2 - 5}{2} \rfloor + 1 = 16
$$
输出通道数：16

**答案**：输出特征图尺寸为 $\boldsymbol{16 \times 16 \times 16}$（通道数 × 高 × 宽）

### 2. 单个输出通道单个像素的点乘次数
单个卷积核尺寸：$3 \times 5 \times 5$，点乘次数等于卷积核参数总数。
$$
3 \times 5 \times 5 = 75
$$

**答案**：需要进行 $\boldsymbol{75}$ 次乘法操作。

In [17]:
import numpy as np

def max_pool2d(x, kernel_size, stride=None, padding=0):
    """
    手动实现二维最大池化前向传播
    参数:
        x: 输入张量，形状 (batch_size, channels, height, width)
        kernel_size: 池化窗口大小 (int 或 tuple)
        stride: 步幅 (int 或 tuple)，默认为 kernel_size
        padding: 填充数 (int)
    返回:
        输出张量
    """
    # 处理参数类型
    if isinstance(kernel_size, int):
        k_h = k_w = kernel_size
    else:
        k_h, k_w = kernel_size
    
    if stride is None:
        s_h = s_w = k_h
    elif isinstance(stride, int):
        s_h = s_w = stride
    else:
        s_h, s_w = stride
    
    if isinstance(padding, int):
        p_h = p_w = padding
    else:
        p_h, p_w = padding
    
    # 获取输入尺寸
    batch, ch, h_in, w_in = x.shape
    
    # 填充
    x_padded = np.pad(x, ((0,0), (0,0), (p_h, p_h), (p_w, p_w)), mode='constant', constant_values=-np.inf)
    h_pad = h_in + 2 * p_h
    w_pad = w_in + 2 * p_w
    
    # 计算输出尺寸
    h_out = (h_pad - k_h) // s_h + 1
    w_out = (w_pad - k_w) // s_w + 1
    
    # 初始化输出
    out = np.empty((batch, ch, h_out, w_out))
    
    # 滑动窗口计算最大值
    for i in range(h_out):
        for j in range(w_out):
            h_start = i * s_h
            h_end = h_start + k_h
            w_start = j * s_w
            w_end = w_start + k_w
            window = x_padded[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2,3))
    
    return out

# 简单测试
if __name__ == "__main__":
    x = np.random.randn(2, 3, 32, 32)
    out = max_pool2d(x, kernel_size=3, stride=2, padding=1)
    print("输入形状:", x.shape)
    print("输出形状:", out.shape)  # 期望 (2,3,16,16) 因为 (32+2-3)//2+1=16

输入形状: (2, 3, 32, 32)
输出形状: (2, 3, 16, 16)


## 3.1 理论计算题
输入输出通道数均为 C，无偏置，参数量 = 卷积核总数量。

### 1. 单个 5×5 卷积层参数量
卷积核：$C_{in}=C,\ C_{out}=C,\ kernel=5\times5$
参数量：
$$
C \times 5 \times 5 \times C = 25C^2
$$
**答案**：$\boldsymbol{25C^2}$

### 2. 两个串联 3×3 卷积层总参数量
第一层：
$$
C \times 3 \times 3 \times C = 9C^2
$$
第二层：
$$
C \times 3 \times 3 \times C = 9C^2
$$
总参数量：
$$
9C^2 + 9C^2 = 18C^2
$$
**答案**：$\boldsymbol{18C^2}$

In [32]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN 块：一个普通卷积 + 两个 1x1 卷积，每层后接 ReLU
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数（用于最后输出）
        kernel_size: 中间卷积层的核大小
        stride: 中间卷积层的步幅
        padding: 中间卷积层的填充
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        # 第一个卷积层：普通卷积
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.relu1 = nn.ReLU()
        # 第二个卷积层：1x1 卷积，不改变空间尺寸，输出通道仍为 out_channels
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.relu2 = nn.ReLU()
        # 第三个卷积层：1x1 卷积
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.relu3 = nn.ReLU()
    
    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.relu3(self.conv3(x))
        return x


class NiNBlockSequential(nn.Module):
    """使用 Sequential 的另一种实现"""
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlockSequential, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)


# 示例：使用 NiNBlock
if __name__ == "__main__":
    # 输入张量: batch=1, 通道=3, 高=32, 宽=32
    x = torch.randn(1, 3, 32, 32)
    
    # 创建 NiN 块：输入3通道，输出96通道，第一个卷积核3x3，步幅1，填充1
    nin_block = NiNBlock(in_channels=3, out_channels=96, kernel_size=3, stride=1, padding=1)
    out = nin_block(x)
    print("使用 NiNBlock:")
    print(f"输入形状: {x.shape} -> 输出形状: {out.shape}")
    # 期望输出形状: (1, 96, 32, 32)
    
    # 也可以使用 Sequential 版本
    nin_seq = NiNBlockSequential(in_channels=3, out_channels=96, kernel_size=3, stride=1, padding=1)
    out_seq = nin_seq(x)
    print("\n使用 NiNBlockSequential:")
    print(f"输入形状: {x.shape} -> 输出形状: {out_seq.shape}")

使用 NiNBlock:
输入形状: torch.Size([1, 3, 32, 32]) -> 输出形状: torch.Size([1, 96, 32, 32])

使用 NiNBlockSequential:
输入形状: torch.Size([1, 3, 32, 32]) -> 输出形状: torch.Size([1, 96, 32, 32])


## 4.1 理论计算题：批量归一化 BN
BN 公式（$\epsilon=0$）：
$$
\mu = \frac{1}{m}\sum_{i=1}^m x_i,\quad \sigma^2 = \frac{1}{m}\sum_{i=1}^m (x_i-\mu)^2
$$
$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}},\quad y_i = \gamma \cdot \hat{x}_i + \beta
$$

已知：$x_1=2,x_2=4,x_3=6,x_4=8,\ m=4,\ \gamma=2,\ \beta=1,\ \epsilon=0$

### 计算均值
$$
\mu = \frac{2+4+6+8}{4} = 5
$$

### 计算方差
$$
\sigma^2 = \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4} = \frac{9+1+1+9}{4} = 5
$$

### 归一化 & 缩放平移
$$
\hat{x}_1=\frac{2-5}{\sqrt{5}},\ \hat{x}_2=\frac{4-5}{\sqrt{5}},\ \hat{x}_3=\frac{6-5}{\sqrt{5}},\ \hat{x}_4=\frac{8-5}{\sqrt{5}}
$$
$$
y_i = 2\cdot \hat{x}_i + 1
$$

### 最终结果
$$
y_1 = 1 - \frac{6}{\sqrt{5}},\quad y_2 = 1 - \frac{2}{\sqrt{5}},\quad y_3 = 1 + \frac{2}{\sqrt{5}},\quad y_4 = 1 + \frac{6}{\sqrt{5}}
$$

小数近似：
$y_1\approx-1.683,\ y_2\approx0.106,\ y_3\approx1.894,\ y_4\approx3.683$

In [28]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块 Residual Block
    :param in_channels: 输入通道数
    :param out_channels: 输出通道数
    :param stride: 卷积步幅
    :param use_1x1conv: 是否使用1x1卷积匹配维度
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super().__init__()
        # 第一个3*3卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # 第二个3*3卷积 + BN
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 1x1卷积：维度对齐
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # 主分支 f(x)
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        #  shortcut 分支
        if self.conv3 is not None:
            x = self.conv3(x)
        # 残差相加 f(x)+x
        y += x
        return self.relu(y)

# 测试
if __name__ == "__main__":
    # 输入 (1, 3, 32, 32)
    x = torch.randn(1, 3, 32, 32)
    # 不使用1x1卷积
    res1 = Residual(3, 3)
    print("无1x1卷积输出shape:", res1(x).shape)
    # 使用1x1卷积，步幅2
    res2 = Residual(3, 16, stride=2, use_1x1conv=True)
    print("有1x1卷积输出shape:", res2(x).shape)

无1x1卷积输出shape: torch.Size([1, 3, 32, 32])
有1x1卷积输出shape: torch.Size([1, 16, 16, 16])


In [20]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块（ResNet basic block）
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数（两个卷积层输出通道数相同）
        use_1x1conv: 是否使用 1x1 卷积调整残差连接
        stride: 第一个卷积层的步幅（默认为1）
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        # 第一个卷积层：3x3，stride 可调
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        # 第二个卷积层：3x3，步幅为1
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 残差连接（跳跃连接）
        if use_1x1conv or stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        residual = self.shortcut(x)   # 调整后的输入
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = self.relu(out)
        return out

# 简单测试
if __name__ == "__main__":
    block = Residual(3, 64, use_1x1conv=True, stride=2)
    x = torch.randn(1, 3, 32, 32)
    y = block(x)
    print("输入形状:", x.shape)
    print("输出形状:", y.shape)  # 期望 (1,64,16,16)

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 64, 16, 16])


## 5.1 理论计算题（简答题）
### 1. 底层特征小学习率、顶层输出层大学习率的原因
预训练模型的底层网络已学习到通用视觉特征（边缘、纹理、轮廓），这类特征具备通用性，无需大幅更新，使用小学习率 / 冻结可以保留有效特征，避免破坏预训练权重。
顶层输出层是针对新任务随机初始化的，和目标任务完全不匹配，需要快速收敛，因此设置更大学习率。

### 2. 数据集小且与源数据集相似的防过拟合微调策略
冻结全部底层卷积层，仅训练最后分类层，完全复用预训练通用特征；
仅使用极小学习率微调少量顶层网络，不改动底层；
配合图像增广、权重衰减、Dropout 等正则化手段；
训练轮数不宜过多，提前停止训练（Early Stopping）。

In [33]:
import torchvision.transforms as transforms
from torchvision.transforms import RandomResizedCrop, RandomHorizontalFlip, ColorJitter, ToTensor

# 创建图像增广管道
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪并缩放到224x224，面积比例0.08~1.0
    RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 2. 50%概率水平翻转
    RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度、饱和度，变化幅度0.5
    ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转换为张量
    ToTensor()
])

# 示例用法（假设有PIL Image）
from PIL import Image
img = Image.open('horse.jpg')
transformed_img = augmentation_pipeline(img)
print(transformed_img.shape)  

torch.Size([3, 224, 224])


## 6.1 理论计算题：IoU 交并比
框格式：$[x_{min}, y_{min}, x_{max}, y_{max}]$

真实框 $A=[10,10,50,50]$
预测框 $B=[30,30,70,70]$

### 步骤 1：计算交集区域
交集坐标：
$x_{min} = \max(10,30)=30,\quad y_{min}=\max(10,30)=30$
$x_{max} = \min(50,70)=50,\quad y_{max}=\min(50,70)=50$

交集宽高：
$w_{inter}=50-30=20,\quad h_{inter}=50-30=20$
交集面积：$S_{inter}=20 \times 20 = 400$

### 步骤 2：计算两个框总面积
$S_A = (50-10)\times(50-10) = 1600$
$S_B = (70-30)\times(70-30) = 1600$

### 步骤 3：计算并集面积
$S_{union} = S_A + S_B - S_{inter} = 1600+1600-400 = 2800$

### 步骤 4：计算 IoU
$IoU = \frac{S_{inter}}{S_{union}} = \frac{400}{2800} = \frac{1}{7}$

**答案**：$\boldsymbol{IoU = \dfrac{1}{7} \approx 0.1429}$

## 6.2 编程题：标签平滑交叉熵损失函数
规则：K 分类，平滑因子 $\epsilon=0.1$

真实类别概率：$1-\epsilon$
其余类别概率：$\dfrac{\epsilon}{K-1}$

In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, num_classes, eps=0.1):
    """
    标签平滑交叉熵损失
    :param logits: 模型原始输出 (N, num_classes)
    :param labels: 真实标签 (N,) 类别索引
    :param num_classes: 分类总数 K
    :param eps: 平滑因子 ε
    :return: 损失值
    """
    # 1. 生成平滑后的目标分布
    soft_target = torch.full_like(logits, fill_value=eps / (num_classes - 1))
    # 真实标签位置赋值 1-eps
    soft_target.scatter_(dim=1, index=labels.unsqueeze(1), value=1 - eps)

    # 2. 计算log_softmax
    log_prob = F.log_softmax(logits, dim=1)

    # 3. 交叉熵 = -sum(soft_target * log_prob)
    loss = -torch.sum(soft_target * log_prob, dim=1).mean()
    return loss


if __name__ == "__main__":
    # 5分类，batch=2
    logits = torch.randn(2, 5)
    labels = torch.tensor([0, 2])
    loss = label_smoothing_cross_entropy(logits, labels, num_classes=5, eps=0.1)
    print("标签平滑损失值：", loss.item())

标签平滑损失值： 2.119999885559082
